In [9]:
import os
# Set timeouts to prevent server disconnection
os.environ["HTTP_TIMEOUT"] = "300"
os.environ["CURL_TIMEOUT"] = "300"

import xarray as xr

# =========================================================================
# 1. Configuration: Models and Run Types
# =========================================================================
models = [
    "CanCM4i", "CanSIPS-IC4", "CanSIPSv2", "CMC1-CanCM3", "CMC2-CanCM4",
    "GEM-NEMO", "GFDL-SPEAR", "NASA-GEOSS2S", "NCAR-CESM1", "NCEP-CFSv2"
]

run_types = ["HINDCAST", "FORECAST"]

# Spatial domain for ATL3 (20°W to 0°)
lon_slice = slice(340, 360) 

# Base directory for outputs
base_output_dir = "/Users/yongyub.kim/kimyy/Research/Postdoc/10_GFI_UiB/UiB/2026_AZM_prediction_LIM/NMME"

# =========================================================================
# 2. Main Loop for Downloads
# =========================================================================
for model in models:
    for run_type in run_types:
        print(f"\n==================================================")
        print(f" Starting Processing: Model [{model}] | Type [{run_type}]")
        print(f"==================================================")
        
        # 2a. Dynamically construct OPeNDAP URL based on IRI Data Library structure
        #     Note: Some models might require minor URL tweaks if the center name differs.
        url = f"https://iridl.ldeo.columbia.edu/SOURCES/.Models/.NMME/.{model}/.{run_type}/.MONTHLY/.sst/dods"
        
        # Define and create a specific directory for each model and run type
        output_dir = f"{base_output_dir}/{model}/SST/ATL3/{run_type}"
        os.makedirs(output_dir, exist_ok=True)
        
        try:
            # 2b. Connect to OPeNDAP server (Metadata only)
            ds = xr.open_dataset(url, decode_times=False)
            
            # 2c. Fix the non-standard 360-day calendar issue for time variables
            for var in ds.variables:
                if 'units' in ds[var].attrs and 'months since' in str(ds[var].attrs['units']):
                    ds[var].attrs['calendar'] = '360_day'
            
            ds = xr.decode_cf(ds, use_cftime=True)
            
            # 2d. Dynamic Y-axis (Latitude) Orientation Check
            #     Handles both South-to-North (-90 to 90) and North-to-South (90 to -90) grids
            if ds.Y[0] > ds.Y[-1]:
                print(f"--> [{model}] Y-axis is North-to-South. Slicing as (3, -3)")
                lat_slice = slice(3, -3)
            else:
                print(f"--> [{model}] Y-axis is South-to-North. Slicing as (-3, 3)")
                lat_slice = slice(-3, 3)
            
            # 2e. Extract available years for the current dataset
            years = sorted(list(set([t.year for t in ds.S.values])))
            print(f"--> Available years: {years[0]} to {years[-1]} (Total {len(years)} years)")
            
            # 2f. Loop through years and save subsetted data
            for year in years:
                file_name = f"{output_dir}/{model}_SST_ATL3_{run_type}_{year}.nc"
                
                # Skip download if the file already exists (fault tolerance)
                if os.path.exists(file_name):
                    continue
                
                print(f"    Processing year {year}...")
                
                # Subset time, longitude, and dynamically adjusted latitude
                year_data = ds.sel(S=(ds['S'].dt.year == year))
                atl3_yearly = year_data.sel(X=lon_slice, Y=lat_slice)
                
                # Save to NetCDF
                atl3_yearly.to_netcdf(file_name)
                
        except Exception as e:
            print(f"\n[ERROR] Failed to process Model: {model} ({run_type}).")
            print(f"Reason: {e}")
            print("Skipping to the next configuration...\n")

print("\n[SUCCESS] All available models and run types have been processed!")


 Starting Processing: Model [CanCM4i] | Type [HINDCAST]


Note:Caching=1
Note:Caching=1


--> [CanCM4i] Y-axis is South-to-North. Slicing as (-3, 3)
--> Available years: 1981 to 2018 (Total 38 years)

 Starting Processing: Model [CanCM4i] | Type [FORECAST]
--> [CanCM4i] Y-axis is South-to-North. Slicing as (-3, 3)
--> Available years: 2016 to 2021 (Total 6 years)

 Starting Processing: Model [CanSIPS-IC4] | Type [HINDCAST]


Note:Caching=1


--> [CanSIPS-IC4] Y-axis is South-to-North. Slicing as (-3, 3)
--> Available years: 1990 to 2024 (Total 35 years)

 Starting Processing: Model [CanSIPS-IC4] | Type [FORECAST]


...

Note:Caching=1


--> [NCAR-CESM1] Y-axis is South-to-North. Slicing as (-3, 3)
--> Available years: 2016 to 2017 (Total 2 years)

 Starting Processing: Model [NCEP-CFSv2] | Type [HINDCAST]


Note:Caching=1


--> [NCEP-CFSv2] Y-axis is South-to-North. Slicing as (-3, 3)
--> Available years: 1982 to 2010 (Total 29 years)

 Starting Processing: Model [NCEP-CFSv2] | Type [FORECAST]

[ERROR] Failed to process Model: NCEP-CFSv2 (FORECAST).
Reason: [Errno -90] NetCDF: file not found: b'https://iridl.ldeo.columbia.edu/SOURCES/.Models/.NMME/.NCEP-CFSv2/.FORECAST/.MONTHLY/.sst/dods'
Skipping to the next configuration...


[SUCCESS] All available models and run types have been processed!


Note:Caching=1
syntax error, unexpected WORD_WORD, expecting SCAN_ATTR or SCAN_DATASET or SCAN_ERROR
context: <html^><head><title>Error 404 Not Found</title></head><body><h1>Error 404: Page Not Found</h1>Error line: 80 anhtmlserverefize 64000000 defre <p>Error on line 0 of stdin: interpreter thinks " .MONTHLY" not defined<form name=stack><table border=2><tr><th><input name=stackdisplay type=checkbox onChange="if(document.getElementById){myobj=document.getElementById('stackdump');if(this.checked){myobj.style.display=''}else{myobj.style.display='none'};};">Show Current Objects</th></tr><tr id="stackdump" style="display: none"><td><ol><li> Ingrid:<li> HTMLwords<li> Models NMME NCEP-CFSv2 FORECAST[<b>EARLY_MONTH_SAMPLES</b><b>PENTAD_SAMPLES</b>]</ol></td></tr></table></form><form method="post" class="ingridcommand" action="/fullcommand.html"><textarea name="command" rows="5" cols="50" wrap="soft"> SOURCES .Models .NMME .NCEP-CFSv2 .FORECAST .MONTHLY .sst</textarea><br /> <input type="submi